# PBL Lab: Understanding Biomedical Data

本notebook离线读取随包提供的四中心教学表，重现Packet B-D证据。它不训练临床模型。


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

ASSETS = Path('../assets/pbl-understanding-data')
data = pd.read_csv(ASSETS / 'heart-disease-four-site.csv')
data['disease_presence'] = (data['num'] > 0).astype(int)
print(data.shape)
data.head()


## A1-A2 / Packet B: structure and quality

先说明一行、一列、目标和site的含义，再查看缺失。


In [ ]:
summary = pd.read_csv(ASSETS / 'dataset-summary.csv')
dictionary = pd.read_csv(ASSETS / 'packet-b-data-dictionary.csv')
site_quality = pd.read_csv(ASSETS / 'packet-b-site-quality.csv')
display(summary)
display(dictionary[['变量','角色','语义类型','缺失率','审查提示']])
display(site_quality)


In [ ]:
missing_by_site = data.groupby('site').apply(lambda g: g.drop(columns=['record_id','site','num','disease_presence']).isna().mean()*100)
display(missing_by_site.round(1))
ax = site_quality.plot(x='中心', y=['缺失率(%)','有病比例(%)'], kind='bar', figsize=(10,4), rot=20)
ax.set_ylabel('Percent'); ax.set_title('Site-specific missingness and outcome composition'); plt.show()


**反思：** site差异可能来自抽样、测量流程、缺失或真实人群差异。当前数据能区分这些解释吗？


## A3 / Packet C: distance and representation sensitivity


In [ ]:
continuous = ['age','trestbps','chol','thalach','oldpeak']
strata = data['site'].astype(str) + '|' + data['disease_presence'].astype(str)
train_i, test_i = train_test_split(np.arange(len(data)), test_size=0.30, random_state=2026, stratify=strata)
train, test = data.iloc[train_i].copy(), data.iloc[test_i].copy()
medians = train[continuous].median()  # training only
Xtr = train[continuous].fillna(medians)
Xte = test[continuous].fillna(medians)
scaler = StandardScaler().fit(Xtr)    # training only
Ztr, Zte = scaler.transform(Xtr), scaler.transform(Xte)
raw = pairwise_distances(Xte, Xtr, metric='euclidean')
scaled = pairwise_distances(Zte, Ztr, metric='euclidean')
changed = (raw.argmin(1) != scaled.argmin(1)).mean()
print(f'Nearest neighbor changed after train-fitted z-score scaling: {changed:.1%}')


In [ ]:
metrics = {
    'raw_euclidean': raw,
    'z_euclidean': scaled,
    'z_manhattan': pairwise_distances(Zte, Ztr, metric='manhattan'),
    'cosine': pairwise_distances(Xte, Xtr, metric='cosine'),
    'correlation': pairwise_distances(Xte, Xtr, metric='correlation'),
}
reference = np.argsort(metrics['z_euclidean'], axis=1)[:, :5]
rows=[]
for name, mat in metrics.items():
    top5=np.argsort(mat,axis=1)[:,:5]
    overlap=np.mean([len(set(top5[i]) & set(reference[i]))/5 for i in range(len(test))])
    rows.append([name, overlap])
pd.DataFrame(rows, columns=['metric','top5_overlap_with_z_euclidean'])


In [ ]:
flags = pd.DataFrame({
 'fbs_high': data['fbs'].eq(1), 'restecg_abnormal': data['restecg'].gt(0),
 'exercise_angina': data['exang'].eq(1), 'asymptomatic_cp': data['cp'].eq(4),
 'non_upsloping': data['slope'].isin([2,3]), 'thal_defect': data['thal'].isin([6,7]),
 'vessels_detected': data['ca'].gt(0)}).astype(int)
print('For sparse presence/absence features, Jaccard ignores joint zeros; SMC counts them.')
pd.read_csv(ASSETS / 'packet-c-binary-similarity.csv')


**反思：** 你要保留的是绝对差异、向量方向、中心化后的形状，还是共同出现？改变语义后，首选度量是否改变？


## A4 / Packet C: sampling and transformation


In [ ]:
replicates = pd.read_csv(ASSETS / 'packet-c-sampling-replicates.csv')
summary_sampling = pd.read_csv(ASSETS / 'packet-c-sampling-summary.csv')
display(summary_sampling)
replicates.boxplot(column='有病比例', by='抽样策略', figsize=(7,4))
plt.suptitle(''); plt.title('Disease prevalence across repeated samples'); plt.show()
display(pd.read_csv(ASSETS / 'packet-c-target-transformation.csv'))


**反思：** 分层保护了哪些已知结构？为什么仍不能声称样本代表目标人群？将num二值化丢失了什么？


## A5-A6 / Packet D: leakage-safe workflow and boundaries


In [ ]:
workflow = pd.read_csv(ASSETS / 'packet-d-preprocessing-audit.csv')
params = pd.read_csv(ASSETS / 'packet-d-parameter-sensitivity.csv')
boundaries = pd.read_csv(ASSETS / 'packet-d-boundary-checklist.csv')
display(workflow)
display(params)
display(boundaries)


In [ ]:
# Programmatic audit: values fitted from the test set must never replace training parameters.
assert set(medians.index) == set(continuous)
assert len(set(train_i) & set(test_i)) == 0
assert len(train_i) + len(test_i) == len(data)
print('Split and train-fitted preprocessing checks passed.')


## Final reflection

1. 你的最终裁决是什么？
2. 引用两项可复现证据。
3. 写出推理链。
4. 明确一个临床、因果或部署边界。
5. 什么新证据会改变你的决定？
